# Backtest & Calibration
Quick backtest on a few races to compute Brier/log loss and reliability for podium/top10 probabilities.

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

from features.feature_engineering import build_features
from simulation.monte_carlo import simulate_probability_table
from simulation.strategy import adjust_mu_for_strategy, adjust_sigma_for_chaos
from models.sigma_model import heuristic_sigma
from utils.session_loader import load_session

In [ ]:
# Races to evaluate (feel free to edit)
races = [
    (2023, "Bahrain", "Q"),
    (2023, "Saudi Arabia", "Q"),
    (2023, "Australia", "Q"),
    (2023, "Azerbaijan", "Q"),
    (2023, "Miami", "Q"),
    (2023, "Monaco", "Q"),
    (2023, "Spain", "Q"),
    (2023, "Canada", "Q"),
    (2023, "Austria", "Q"),
]
results = []


In [ ]:
for year, event, session_name in races:
    session_data = load_session(year, event, session_name)
    laps = session_data["laps"]
    event_name = session_data["session"].event.EventName
    features_df = build_features(laps, event_name=event_name)
    mu = -features_df["median_lap_s"].to_numpy()
    sigma = heuristic_sigma(features_df)
    mu = adjust_mu_for_strategy(mu, features_df)
    sigma = adjust_sigma_for_chaos(sigma, features_df)
    drivers = features_df["Driver"].tolist()

    sim = simulate_probability_table(mu, sigma, n_sims=3000, driver_labels=drivers, random_state=42)
    # Observed: use qualifying position as proxy for finish in this backtest
    observed = features_df[["Driver", "fastest_lap"]].sort_values("fastest_lap").reset_index(drop=True)
    observed["finish_pos"] = observed.index + 1
    obs_map = dict(zip(observed.Driver, observed.finish_pos))

    # Brier score for top10
    top10 = sim.probabilities[[f"P{i}" for i in range(1, 11)]].sum(axis=1)
    y_true = drivers.copy()
    y_true = [1 if obs_map[d] <= 10 else 0 for d in drivers]
    brier = np.mean((top10.to_numpy() - np.array(y_true)) ** 2)

    # Expected finish vs observed (MAE)
    exp_finish = sim.expected_finish
    mae = np.mean(np.abs(exp_finish - np.array([obs_map[d] for d in drivers])))

    results.append({
        "year": year,
        "event": event,
        "session": session_name,
        "brier_top10": brier,
        "mae_finish": mae,
    })

backtest_df = pd.DataFrame(results)
backtest_df

In [ ]:
fig = px.bar(backtest_df, x="event", y=["brier_top10", "mae_finish"], barmode="group", title="Backtest metrics")
fig.show()